<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/choose_source1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
# 1) Data input

import os
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# Mount Drive only once at the beginning
drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

# Global variables — user selection result
chosen_source = None
chosen_path = None

def choose_source() -> (str, str):
    """
    parameters: none
    purpose: asks the user from which source they want to load data (csv, json, api)
    and for the file path or API URL; both selections are saved into the variables
    source and path, and then returned
    output: source, path — text describing the type of source and data location
    """
    global chosen_source, chosen_path

    # --- UI elements ---
    fmt_label = widgets.Label("Select data format:")
    fmt_dd = widgets.Dropdown(
        options=[('CSV', 'csv'), ('JSON', 'json'), ('API', 'api')],
        description='Format:',
        value='csv'
    )

    src_label = widgets.Label("Select data source:")
    src_dd = widgets.Dropdown(
        options=[('File path', 'path'), ('Upload from computer', 'upload')],
        description='Source:',
        value='path'
    )

    file_path = widgets.Text(placeholder='Enter file path...', description='Path:')
    api_url = widgets.Text(placeholder='Enter API URL...', description='URL:')
    upload_btn = widgets.FileUpload(description='Upload file', multiple=False)
    ok_btn = widgets.Button(description="Confirm")
    out = widgets.Output()

    # --- hide fields initially ---
    src_dd.layout.display = 'none'
    file_path.layout.display = 'none'
    upload_btn.layout.display = 'none'
    api_url.layout.display = 'none'

    # --- reactions to changes ---
    def on_fmt_change(change):
        src_dd.layout.display = 'none'
        file_path.layout.display = 'none'
        upload_btn.layout.display = 'none'
        api_url.layout.display = 'none'
        if change['new'] in ['csv', 'json']:
            src_dd.layout.display = 'block'
        elif change['new'] == 'api':
            api_url.layout.display = 'block'

    def on_src_change(change):
        file_path.layout.display = 'none'
        upload_btn.layout.display = 'none'
        if change['new'] == 'path':
            file_path.layout.display = 'block'
        elif change['new'] == 'upload':
            upload_btn.layout.display = 'block'

    fmt_dd.observe(on_fmt_change, names='value')
    src_dd.observe(on_src_change, names='value')

    # --- after clicking Confirm ---
    def on_ok_click(b):
        global chosen_source, chosen_path
        out.clear_output()
        fmt = fmt_dd.value
        src_type = src_dd.value

        try:
            if fmt in ['csv', 'json']:
                if src_type == 'path':
                    path = file_path.value.strip()
                    if not path:
                        raise ValueError("Please enter a file path.")
                    if not os.path.exists(path):
                        raise FileNotFoundError(f"File not found: {path}")
                elif src_type == 'upload':
                    if not upload_btn.value:
                        raise ValueError("Please upload a file first.")
                    file = list(upload_btn.value.values())[0]
                    ext = '.csv' if fmt == 'csv' else '.json'
                    path = f"/content/drive/MyDrive/ml_project/upload{ext}"
                    # Remove old uploaded files before saving a new one
                    for ext_old in ('.csv', '.json'):
                        old = f"/content/drive/MyDrive/ml_project/upload{ext_old}"
                        if os.path.exists(old):
                            os.remove(old)
                    with open(path, "wb") as f:
                        f.write(file['content'])
                chosen_source = fmt
                chosen_path = path
                print(f"Selected source: {chosen_source}")
                print(f"Path: {chosen_path}")

            elif fmt == 'api':
                url = api_url.value.strip()
                if not url:
                    raise ValueError("Please enter an API URL.")
                chosen_source = 'api'
                chosen_path = url
                print(f"Selected API: {chosen_path}")

        except Exception as e:
            print(f"Error: {e}")

    ok_btn.on_click(on_ok_click)

    # --- display UI ---
    display(widgets.VBox([
        fmt_label, fmt_dd,
        src_label, src_dd,
        file_path, api_url, upload_btn,
        ok_btn, out
    ]))


# FUNCTION CALL
print("\n1. Choosing data source...")
choose_source()


Mounted at /content/drive

1. Choosing data source...


Selected source: csv
Path: /content/drive/MyDrive/ml_project/upload.csv
